# 03 — LoRA adaptation

**Purpose:** Train adapter-only versions of the same Qwen checkpoint; save logs, trained-parameter count, dataset hashes and adapter files.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


### Training choices

Default: `10 examples/class`, rank 8, seed 42, two epochs. After the pilot works, add `25,50` and seed repeats. All training uses only the training split. Validation is available for selection but the shared training function does not read the test set.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU on Kaggle/Colab. CPU fine-tuning is intentionally blocked here.'
TASK='sib200'; LANG='ur'
SIZES=[10]  # after pilot, [10,25,50]
RANKS=[8]  # after pilot, [8,16] at a fixed training size
SEEDS=[42] # after pilot, [42,123,456]
MODEL_OVERRIDE = None  # set to saved English anchor path for controlled forgetting
DRY_RUN = False
print('GPU:',torch.cuda.get_device_name(0))
for seed in SEEDS:
    for n in SIZES:
        for rank in RANKS:
            run_name = f'{TASK}_lora_{LANG}_n{n}_s{seed}_r{rank}' + ('_anchor' if MODEL_OVERRIDE else '')
            if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
                print('Already complete; skip:',run_name); continue
            if DRY_RUN: print('Would run:',run_name); continue
            r=train_run('lora',TASK,n,seed,run_name=run_name,lang=LANG,rank=rank,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),
                  'peak GiB:',r['peak_gpu_allocated_gib'],'trainable %:',r['trainable_pct'])

The checkpoint contains **LoRA adapters**, not a standalone copy of the full Qwen model. For inference the base model and adapter must both be loaded. Do not combine adapters from different base checkpoints.